# 01 — Root finding: convergence-rate visualization

Drives `BisectionSolver`, `NewtonSolver`, `SecantSolver`, `BrentSolver` on the same test function and plots $|e_k|$ vs $k$ on a log-y axis.

Math reminders:
- Bisection: $|e_{k+1}| = \tfrac12|e_k|$ (order 1)
- Newton: $|e_{k+1}| \sim |e_k|^2$ (order 2)
- Secant: order $\varphi \approx 1.618$
- Brent: superlinear with bisection-safe worst case

**Prerequisites**: implement `IterativeRootFinder::solve()` and `CapturingObserver::on_iteration()`.

In [ ]:
import sys
from pathlib import Path
for so in Path('../build').rglob('nmpy*.so'):
    sys.path.insert(0, str(so.parent))
    break
import nmpy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
f  = lambda x: x*x - 2.0
df = lambda x: 2.0*x

def run(kind, **kwargs):
    cfg = nmpy.SolverConfig()
    cfg.f = f
    cfg.df = df
    cfg.a, cfg.b, cfg.x0, cfg.x1 = 0.0, 2.0, 1.0, 2.0
    s = nmpy.create_solver(kind, cfg)
    cap = nmpy.CapturingObserver()
    s.attach(cap)
    r = s.solve(1e-14, 200)
    return r, cap

In [ ]:
fig, ax = plt.subplots()
for kind, label in [(nmpy.SolverKind.BISECTION, 'bisection'),
                    (nmpy.SolverKind.NEWTON,    'newton'),
                    (nmpy.SolverKind.SECANT,    'secant'),
                    (nmpy.SolverKind.BRENT,     'brent')]:
    r, _ = run(kind)
    ax.semilogy(r.errors, label=f'{label} (p~{nmpy.estimate_order(r.errors):.2f})')
ax.set(xlabel='iteration k', ylabel='|e_k|', title='Convergence rates')
ax.legend(); plt.show()